# Demo de `toolbox_ml`

Este notebook demuestra el uso de todas las funciones del paquete `toolbox_ml.eda.core` sobre un dataset real: el clásico dataset del Titanic.

**Requisito previo:** el paquete debe estar instalado en modo editable desde la raíz del repositorio:

```bash
pip install -r requirements.txt
pip install -e .
```

In [ ]:
import pandas as pd

from toolbox_ml.eda.core import (
    describe_df,
    tipifica_variables,
    get_features_num_regression,
    plot_features_num_regression,
    get_features_cat_regression,
    plot_features_cat_regression,
    detect_outliers,
)

df = pd.read_csv("titanic.csv")
df.head()

## 1. `describe_df`

Genera un resumen rápido de cada columna: tipo de dato, porcentaje de nulos, número de valores únicos y porcentaje de cardinalidad.

In [ ]:
describe_df(df)

## 2. `tipifica_variables`

Sugiere automáticamente qué tipo de variable es cada columna (Binaria, Categórica, Numérica Continua o Numérica Discreta), en función de dos umbrales que definimos nosotros.

In [ ]:
tipifica_variables(df, umbral_categoria=10, umbral_continua=30.0)

## 3. `get_features_num_regression` y `plot_features_num_regression`

Usamos `fare` (precio del billete) como variable objetivo numérica de ejemplo, y buscamos qué otras columnas numéricas están correlacionadas con ella.

In [ ]:
columnas_num_relevantes = get_features_num_regression(
    df, target_col="fare", umbral_corr=0.1, pvalue=0.05
)
print("Columnas numéricas seleccionadas:", columnas_num_relevantes)

In [ ]:
# plot_features_num_regression repite el filtro y además pinta un pairplot
_ = plot_features_num_regression(
    df, target_col="fare", umbral_corr=0.1, pvalue=0.05
)

## 4. `get_features_cat_regression` y `plot_features_cat_regression`

Buscamos qué columnas categóricas (`sex`, `embarked`, `class`, `who`...) tienen una relación estadísticamente significativa con `fare`.

In [ ]:
columnas_cat_relevantes = get_features_cat_regression(df, target_col="fare", pvalue=0.05)
print("Columnas categóricas seleccionadas:", columnas_cat_relevantes)

In [ ]:
_ = plot_features_cat_regression(
    df, target_col="fare", pvalue=0.05, with_individual_plot=False
)

## 5. BONUS: `detect_outliers`

Detectamos outliers en las columnas numéricas usando dos métodos (IQR y Z-score) y comparamos cuántos detecta cada uno.

In [ ]:
outliers = detect_outliers(df)

for columna, resultado in outliers.items():
    print(
        f"{columna:10s} | IQR: {resultado['iqr']['n_outliers']:3d} "
        f"({resultado['iqr']['porcentaje']:5.2f}%) | "
        f"Z-score: {resultado['zscore']['n_outliers']:3d} "
        f"({resultado['zscore']['porcentaje']:5.2f}%)"
    )

In [ ]:
# Ejemplo: primeros índices de los outliers detectados en 'fare' por IQR
outliers["fare"]["iqr"]["indices"][:10]

## Conclusión

Con estas siete funciones se cubre un flujo de EDA razonablemente completo: descripción y tipificación automática de variables, selección de features numéricas y categóricas relevantes para un problema de regresión (con su visualización correspondiente), y detección de outliers. Todo el código vive en `toolbox_ml/eda/core.py`, está testeado en `tests/test_core.py` y es reutilizable en cualquier notebook o script simplemente con:

```python
from toolbox_ml.eda.core import describe_df, tipifica_variables, ...
```